# XGBoost 주가 예측기 (앙상블 + Optuna) — 빈칸 연습

**원본**: `notebooks/XGBoost_Predict.ipynb`

`___` 빈칸을 채워서 코드를 완성하세요.

30년치 미국 주식 데이터를 앙상블 모델로 학습해 향후 30일 주가를 재귀적으로 예측합니다.

- **입력**: 기술적 지표(수익률·이동평균·RSI·거래량비율) + 외부 변수(VIX·금리) + 섹터 ETF 수익률
- **튜닝**: Optuna로 XGBoost·LightGBM·CatBoost 하이퍼파라미터 자동 탐색 (각 100회)
- **앙상블**: XGBoost + LightGBM + CatBoost 예측 → Ridge 메타 러너
- **예측**: 다음날 수익률 → 30일간 반복 적용 → 가격으로 환산

## Q1. 라이브러리 임포트 (빈칸 7개)
- XGBoost / LightGBM / CatBoost 회귀 모델 클래스명은?
- 피처 정규화에 사용하는 sklearn 클래스명은?
- 평가 지표 MAE 함수명은?


In [ ]:
# 환경 설정 — 시스템 경로 및 경고 억제
import sys  # 시스템 모듈 임포트
sys.path.insert(0, '..')  # 상위 디렉토리를 모듈 경로에 추가

import warnings  # 경고 메시지 관리 모듈
warnings.filterwarnings('ignore')  # 모든 경고 메시지 숨김

import numpy as np  # 수치 연산 라이브러리
import pandas as pd  # 데이터프레임 라이브러리
import matplotlib.pyplot as plt  # 시각화 라이브러리
import matplotlib  # matplotlib 설정용
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
import requests  # HTTP 요청 라이브러리
import datetime  # 날짜/시간 처리 모듈

In [ ]:
# 머신러닝 모델 및 도구 임포트
from xgboost import ___  # Q1-1: XGBoost 회귀 모델 클래스
from lightgbm import ___  # Q1-2: LightGBM 회귀 모델 클래스
from catboost import ___  # Q1-3: CatBoost 회귀 모델 클래스
from sklearn.ensemble import RandomForestRegressor  # 랜덤포레스트 회귀 모델
from sklearn.preprocessing import ___  # Q1-4: 표준 정규화 스케일러
from sklearn.metrics import ___  # Q1-5: 평균 절대 오차 함수

import optuna  # 하이퍼파라미터 자동 튜닝 라이브러리
optuna.logging.set_verbosity(optuna.logging.WARNING)  # Optuna 로그 레벨을 경고로 설정

from sklearn.linear_model import ___  # Q1-6: 릿지 회귀 (메타 러너용)

from dotenv import load_dotenv  # 환경변수 파일 로더
load_dotenv('../.env')  # .env 파일에서 API 키 등 로드

print('환경 설정 완료')  # 설정 완료 확인 메시지

### 데이터 수집
1. `fetch_macro`를 활용해서 주가 및 기술 데이터 수집
2. 섹터 ETF(XLF·XLE·XLK·XLV·GLD) 30년치 수집

In [ ]:
from collector.market_data import fetch_macro  # 거시지표 수집 함수 임포트

# S&P 500 기준으로 30년치 데이터 수집
df_raw = fetch_macro()  # 거시 데이터 가져오기
# 인덱스의 첫번째 date ~ 마지막 date 출력
print(f'수집 기간: {df_raw.index[0].date()} ~ {df_raw.index[-1].date()}')
print(f'shape: {df_raw.shape}')  # 데이터 크기 확인
df_raw[['sp500_close', 'sp500_return', 'sp500_rsi', 'vix', 'tnx']].tail(3)  # 최근 3행 미리보기

## Q2. 섹터 ETF 데이터 수집 (빈칸 1개)
- 유닉스 타임스탬프를 날짜로 변환할 때 단위는?


In [ ]:
SECTOR_TICKERS = ['XLF', 'XLE', 'XLK', 'XLV', 'GLD']  # 섹터 ETF 티커 목록
HEADERS = {'User-Agent': 'Mozilla/5.0'}  # HTTP 요청 헤더 (봇 차단 방지)

today     = datetime.date.today()  # 오늘 날짜
from_date = today - datetime.timedelta(days=365 * 30)  # 30년 전 날짜
epoch     = datetime.datetime(1970, 1, 1)  # 유닉스 에포크 기준
# 시작 날짜를 유닉스 타임스탬프로 변환
from_ts   = int((datetime.datetime.combine(from_date, datetime.time()) - epoch).total_seconds())
# 종료 날짜를 유닉스 타임스탬프로 변환
to_ts     = int((datetime.datetime.combine(today, datetime.time()) - epoch).total_seconds())

sector_raw = {}  # 섹터별 종가 데이터 저장 딕셔너리
# 섹터 티커들을 순회하며 데이터 수집
for ticker in SECTOR_TICKERS:
    # Yahoo Finance API URL 구성
    url    = f'https://query1.finance.yahoo.com/v8/finance/chart/{ticker}'
    params = {'interval': '1d', 'period1': from_ts, 'period2': to_ts}  # 일별 데이터 요청 파라미터
    resp   = requests.get(url, params=params, headers=HEADERS, timeout=15)  # API 호출
    result = resp.json()['chart']['result'][0]  # JSON 응답에서 차트 데이터 추출
    closes = result['indicators']['adjclose'][0]['adjclose']  # 수정 종가 리스트 추출
    # 타임스탬프를 날짜 인덱스로 변환
    index  = pd.to_datetime(result['timestamp'], unit='___').normalize()  # Q2-1: 유닉스 타임스탬프 단위
    # 종가 데이터를 판다스 시리즈로 저장
    sector_raw[ticker] = pd.Series(closes, index=index, name=ticker)
    print(f'{ticker}: {len(sector_raw[ticker])}행')  # 수집 건수 출력

### 피처 엔지니어링

| 피처 | 설명 |
|------|------|
| `sp500_return` | 전일 대비 수익률 |
| `sp500_return_5` | 5일 누적 수익률 |
| `sp500_return_20` | 20일 누적 수익률 |
| `sp500_ma20_disp` | 20일 이동평균 괴리율 |
| `sp500_rsi` | 14일 RSI |
| `sp500_vol_ratio` | 거래량/20일 평균 |
| `vix` | 변동성 지수 |
| `tnx` | 10년물 금리 |
| `yield_spread` | 장단기 금리차 |
| `xlf/xle/xlk/xlv/gld_return` | 섹터 ETF 수익률 |
| `xlk_sp500_rs` | 기술주 상대 강도 (XLK - S&P500 5일 수익률) |

## Q3. 수익률 피처 생성 (빈칸 6개)
- 5일/20일 누적 수익률: `pct_change(?)`
- 섹터 ETF 1일 수익률: `pct_change(?)`
- 기술주 상대 강도: XLK 5일 수익률 - S&P500 5일 수익률
- 타깃: 다음날 수익률 = `shift(?)`


In [ ]:
df = df_raw.copy()  # 원본 보존을 위해 복사본 생성

# 5일 누적 수익률 피처 생성
df['sp500_return_5']  = df['sp500_close'].pct_change(___)  # Q3-1: 5일 누적 수익률 기간
# 20일 누적 수익률 피처 생성
df['sp500_return_20'] = df['sp500_close'].pct_change(___)  # Q3-2: 20일 누적 수익률 기간

# 섹터 ETF 1일 수익률 계산
for ticker in SECTOR_TICKERS:
    name = ticker.lower()  # 티커명을 소문자로 변환
    s = sector_raw[ticker].reindex(df.index).ffill()  # 인덱스 맞추고 결측치 전방 채움
    df[f'{name}_return'] = s.pct_change(___)  # Q3-3: 1일 수익률 기간

In [ ]:
# 기술주 상대 강도 (XLK 5일 수익률 - S&P500 5일 수익률)
df['xlk_sp500_rs'] = (
    sector_raw['XLK'].reindex(df.index).ffill().pct_change(___)  # Q3-4: XLK 누적 수익률 기간
    - df['sp500_close'].pct_change(___)  # Q3-5: S&P500 누적 수익률 기간
)

# 예측 타깃: 다음날 수익률 (shift -1 = 미래 값을 현재 행으로 당김)
df['target'] = df['sp500_return'].shift(___)  # Q3-6: 다음날 수익률을 가져오는 shift 값

# 결측치 제거 (피처 생성 과정에서 발생한 NaN 삭제)
df = df.dropna()  # NaN이 포함된 행 제거

In [ ]:
# 학습 피처 칼럼 목록 정의 (중요도 하위 4개 제거: sp500_return, sp500_return_5, sp500_rsi, gld_return)
FEATURE_COLS = [
    'sp500_return_20',  # 20일 누적 수익률
    'sp500_ma20_disp', 'sp500_vol_ratio',  # MA20 괴리율, 거래량 비율
    'vix', 'tnx', 'yield_spread', 'dxy_return',  # VIX, 금리, 금리차, 달러 수익률
    'xlf_return', 'xle_return', 'xlk_return', 'xlv_return',  # 섹터 ETF 수익률
    'xlk_sp500_rs',  # 기술주 상대 강도
]

print(f'학습 데이터: {len(df)}행 × {len(FEATURE_COLS)}개 피처')  # 데이터 크기 확인
df[FEATURE_COLS].tail(3)  # 최근 3행 미리보기

## Q4. 라그 / 변동성 / 계절성 피처 (빈칸 10개)
- 라그 피처: `shift(?)` — 2일 전, 5일 전 VIX·수익률
- 변동성: `rolling(?).std()` — 20일, 60일 수익률 표준편차
- 시장 레짐: 현재가 > `rolling(?).mean()` 이면 상승장(1)


In [ ]:
# 라그 피처 (t-2, t-5일 전 VIX·수익률 → 모멘텀 패턴 학습)
df['vix_lag2'] = df['vix'].shift(___)  # Q4-1: VIX 2일 전 값
df['vix_lag5'] = df['vix'].shift(___)  # Q4-2: VIX 5일 전 값
df['ret_lag2'] = df['sp500_return'].shift(___)  # Q4-3: 수익률 2일 전 값
df['ret_lag5'] = df['sp500_return'].shift(___)  # Q4-4: 수익률 5일 전 값

In [ ]:
# 변동성 피처 (단기·중기 수익률 표준편차 → 고변동 구간 인식)
df['vol_20d'] = df['sp500_return'].rolling(___).___()  # Q4-5,6: 20일 롤링 표준편차
df['vol_60d'] = df['sp500_return'].rolling(___).___()  # Q4-7,8: 60일 롤링 표준편차

# 계절성 피처 (요일: 0=월~4=금, 월: 1~12 → 월말·연말 효과 학습)
df['day_of_week'] = df.index.dayofweek  # 요일 피처
df['month']       = df.index.month  # 월 피처

# 시장 레짐 판별 (MA200 기준: 현재가 > 200일 이동평균이면 상승장)
df['is_bull'] = (df['sp500_close'] > df['sp500_close'].rolling(___).___()).astype(int)  # Q4-9,10: 200일 이동평균

In [ ]:
df = df.dropna()  # 라그/롤링 계산으로 생긴 NaN 제거

# 기존 피처 목록에 새 피처 추가
FEATURE_COLS = FEATURE_COLS + [
    'vix_lag2', 'vix_lag5', 'ret_lag2', 'ret_lag5',  # 라그 피처
    'vol_20d', 'vol_60d',  # 변동성 피처
    'day_of_week', 'month', 'is_bull',  # 계절성 + 레짐 피처
]

print(f'피처 추가 후: {len(FEATURE_COLS)}개 피처')  # 피처 수 확인
df[['vix_lag2', 'vol_20d', 'day_of_week', 'is_bull']].tail(3)  # 새 피처 미리보기

## Q5. 타깃 이상치 클리핑 (빈칸 4개)
- 극단값 제거 기준: ±?σ (σ = 표준편차)
- `clip(lower=평균 - ?×σ, upper=평균 + ?×σ)`


In [ ]:
# 타깃 이상치 클리핑 (극단값을 ±3σ 범위로 잘라내어 학습 왜곡 방지)
ret_mean = df['target'].___()  # Q5-1: 타깃의 평균값 계산
ret_std  = df['target'].___()  # Q5-2: 타깃의 표준편차 계산
# ±3σ 범위로 클리핑
df['target'] = df['target'].clip(
    lower=ret_mean - ___*ret_std,  # Q5-3: 하한 클리핑 배수
    upper=ret_mean + ___*ret_std   # Q5-4: 상한 클리핑 배수
)
print(f'클리핑 범위: [{ret_mean - 3*ret_std:.4f}, {ret_mean + 3*ret_std:.4f}]')
print(f'타깃 통계  | 평균: {df["target"].mean():.6f}, 표준편차: {df["target"].std():.6f}')

### 학습 / 검증 분할 및 앙상블 학습

- 마지막 252일(약 1년)을 검증셋으로 사용
- 나머지를 학습셋으로 사용
- Optuna로 XGBoost·LightGBM·CatBoost 파라미터 자동 탐색 (각 100회)
- 스태킹: Ridge 메타 러너로 최종 예측

## Q6. 학습/검증 분할 및 스케일링 (빈칸 5개)
- 검증 기간 = 약 1년 = ?영업일
- 스케일러 클래스? 학습 데이터에는 `fit_transform`, 검증 데이터에는 `transform`
- 지수 감쇠 계수?


In [ ]:
VALID_DAYS = ___  # Q6-1: 검증 기간 (약 1년 = 252 영업일)

X = df[FEATURE_COLS].values  # 피처 행렬 (numpy 배열)
y = df['target'].values  # 타깃 벡터 (numpy 배열)

# 시계열 순서를 유지한 train/valid 분할 (미래 데이터 누수 방지)
X_train, X_valid = X[:-VALID_DAYS], X[-VALID_DAYS:]  # 훈련은 앞 / 검증은 뒤
y_train, y_valid = y[:-VALID_DAYS], y[-VALID_DAYS:]  # 타깃도 동일하게 분할

In [ ]:
# 피처 정규화 (평균=0, 분산=1로 변환)
scaler    = ___()  # Q6-2: 표준 정규화 스케일러 생성
X_train_s = scaler.___(X_train)  # Q6-3: 학습 데이터에 맞춰 정규화 학습+변환
X_valid_s = scaler.___(X_valid)  # Q6-4: 검증 데이터는 학습 통계로 변환만

# 최근 데이터에 더 높은 가중치 부여 (지수 감쇠)
decay = ___  # Q6-5: 지수 감쇠 계수 (1에 가까울수록 완만한 감쇠)
# 가장 오래된 데이터 → 가장 작은 가중치, 최신 데이터 → 1.0
sample_weight = decay ** np.arange(len(X_train) - 1, -1, -1)

print(f'Train: {len(X_train)}행, Valid: {len(X_valid)}행')
print(f'샘플 가중치 범위: {sample_weight.min():.4f} ~ {sample_weight.max():.4f}')

## Q7. Optuna 하이퍼파라미터 튜닝 (빈칸 8개)
- 모델 학습: `.fit(X, y, sample_weight=...)`
- 평가 함수: MAE 함수명?
- 최적화 방향: MAE를 최소화 → `direction='?'`


In [ ]:
# XGBoost Optuna 목적 함수 정의
def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1500),  # 부스팅 라운드 수
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),  # 학습률
        'max_depth':        trial.suggest_int('max_depth', 3, 7),  # 트리 최대 깊이
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),  # 행 샘플링 비율
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),  # 열 샘플링 비율
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),  # 리프 노드 최소 가중치
        'gamma':            trial.suggest_float('gamma', 0.0, 0.3),  # 분할 최소 손실 감소량
        'random_state': 42,  # 재현성을 위한 랜덤 시드
        'verbosity': 0,  # 로그 출력 끔
    }
    m = XGBRegressor(**params)  # XGBoost 회귀 모델 생성
    m.___(X_train_s, y_train, sample_weight=sample_weight)  # Q7-1: 모델 학습
    return ___(y_valid, m.___(X_valid_s))  # Q7-2,3: MAE 계산 (정답, 예측)

In [ ]:
# LightGBM Optuna 목적 함수 정의
def objective_lgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),  # 부스팅 라운드 수
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),  # 학습률
        'max_depth':         trial.suggest_int('max_depth', 3, 7),  # 트리 최대 깊이
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),  # 행 샘플링 비율
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),  # 열 샘플링 비율
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),  # 리프 노드 최소 샘플 수
        'num_leaves':        trial.suggest_int('num_leaves', 20, 100),  # 트리의 최대 리프 수
        'random_state': 42,  # 재현성을 위한 랜덤 시드
        'verbose': -1,  # 로그 출력 끔
    }
    m = LGBMRegressor(**params)  # LightGBM 회귀 모델 생성
    m.___(X_train_s, y_train, sample_weight=sample_weight)  # Q7-4: 모델 학습
    return ___(y_valid, m.___(X_valid_s))  # Q7-5: MAE 평가

In [ ]:
# CatBoost Optuna 목적 함수 정의
def objective_cat(trial):
    params = {
        'iterations':    trial.suggest_int('iterations', 300, 1500),  # 부스팅 반복 횟수
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),  # 학습률
        'depth':         trial.suggest_int('depth', 3, 8),  # 트리 깊이
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1.0, 10.0),  # L2 정규화 계수
    }
    m = CatBoostRegressor(**params, random_seed=42, verbose=0)  # CatBoost 모델 생성
    m.___(X_train_s, y_train, sample_weight=sample_weight)  # Q7-6: 모델 학습
    return ___(y_valid, m.___(X_valid_s))  # Q7-7: MAE 평가

## Q8. Optuna 스터디 실행 (빈칸 3개)
- MAE를 최소화하는 방향: `direction='?'`


In [ ]:
# Optuna로 XGBoost 하이퍼파라미터 탐색
print('XGBoost 튜닝 중...')
study_xgb = optuna.create_study(direction='___')  # Q8-1: 최적화 방향 (MAE 최소화)
study_xgb.optimize(objective_xgb, n_trials=100)  # 100회 탐색 실행

# Optuna로 LightGBM 하이퍼파라미터 탐색
print('LightGBM 튜닝 중...')
study_lgb = optuna.create_study(direction='___')  # Q8-2: 최적화 방향
study_lgb.optimize(objective_lgb, n_trials=100)  # 100회 탐색 실행

# Optuna로 CatBoost 하이퍼파라미터 탐색
print('CatBoost 튜닝 중...')
study_cat = optuna.create_study(direction='___')  # Q8-3: 최적화 방향
study_cat.optimize(objective_cat, n_trials=100)  # 100회 탐색 실행

# 각 모델의 최적 파라미터 및 성능 출력
print('\nXGBoost  최적 파라미터:', study_xgb.best_params)
print(f'XGBoost  Best MAE: {study_xgb.best_value:.6f}')
print()
print('LightGBM 최적 파라미터:', study_lgb.best_params)
print(f'LightGBM Best MAE: {study_lgb.best_value:.6f}')
print()
print('CatBoost 최적 파라미터:', study_cat.best_params)
print(f'CatBoost Best MAE: {study_cat.best_value:.6f}')

## Q9. 앙상블 모델 학습 (빈칸 6개)
- 최적 파라미터로 3개 모델을 각각 학습
- `.fit(X, y, sample_weight=...)`


In [ ]:
# XGBoost 최적 파라미터로 최종 모델 학습
model_xgb = ___(  # Q9-1: XGBoost 회귀 모델 클래스
    **study_xgb.best_params, random_state=42, verbosity=0
)
model_xgb.___(X_train_s, y_train, sample_weight=sample_weight)  # Q9-2: 모델 학습

# LightGBM 최적 파라미터로 최종 모델 학습
model_lgb = ___(  # Q9-3: LightGBM 회귀 모델 클래스
    **study_lgb.best_params, random_state=42, verbose=-1
)
model_lgb.___(X_train_s, y_train, sample_weight=sample_weight)  # Q9-4: 모델 학습

# CatBoost 최적 파라미터로 최종 모델 학습
model_cat = ___(  # Q9-5: CatBoost 회귀 모델 클래스
    **study_cat.best_params, random_seed=42, verbose=0
)
model_cat.___(X_train_s, y_train, sample_weight=sample_weight)  # Q9-6: 모델 학습

print('모델 학습 완료')

## Q10. MAE 역수 가중 앙상블 (빈칸 9개)
- 검증셋 예측: `model.predict(?)`
- 개별 모델 MAE 계산
- MAE 역수 가중치: `w = 1 / mae` (MAE가 낮을수록 높은 가중치)
- 가중 평균: `(w_xgb×pred_xgb + w_lgb×pred_lgb + w_cat×pred_cat) / total`


In [ ]:
# 검증셋에 대한 각 모델의 예측값 계산
pred_xgb = model_xgb.___(X_valid_s)  # Q10-1: XGBoost 예측
pred_lgb = model_lgb.___(X_valid_s)  # Q10-2: LightGBM 예측
pred_cat = model_cat.___(X_valid_s)  # Q10-3: CatBoost 예측

# 개별 모델의 MAE(평균 절대 오차) 계산
mae_xgb = ___(y_valid, pred_xgb)  # Q10-4: XGBoost MAE
mae_lgb = ___(y_valid, pred_lgb)  # Q10-5: LightGBM MAE
mae_cat = ___(y_valid, pred_cat)  # Q10-6: CatBoost MAE

In [ ]:
# MAE 역수 가중 앙상블 (MAE가 낮은 모델에 더 높은 가중치 부여)
w_xgb = ___ / mae_xgb  # Q10-7: MAE 역수 가중치 (분자)
w_lgb = ___ / mae_lgb  # Q10-8: MAE 역수 가중치 (분자)
w_cat = ___ / mae_cat  # Q10-9: MAE 역수 가중치 (분자)
total = w_xgb + w_lgb + w_cat  # 가중치 합계 (정규화용)

# 가중 평균으로 앙상블 예측값 계산
pred_ens = (w_xgb*pred_xgb + w_lgb*pred_lgb + w_cat*pred_cat) / total
mae_ens  = mean_absolute_error(y_valid, pred_ens)  # 앙상블 MAE 계산

print(f'XGBoost  MAE: {mae_xgb:.6f} ({mae_xgb*100:.4f}%)')
print(f'LightGBM MAE: {mae_lgb:.6f} ({mae_lgb*100:.4f}%)')
print(f'CatBoost MAE: {mae_cat:.6f} ({mae_cat*100:.4f}%)')
print(f'앙상블   MAE: {mae_ens:.6f} ({mae_ens*100:.4f}%)')
print(f'\n가중치 | XGB:{w_xgb/total:.3f} LGB:{w_lgb/total:.3f} CAT:{w_cat/total:.3f}')

## Q11. 스태킹 메타 러너 (빈칸 7개)
- Ridge 회귀로 3모델 예측을 결합
- `positive=True` → 음수 가중치 방지
- 방향 정확도: `np.sign()` 으로 부호 비교


In [ ]:
# 스태킹: 검증셋 전반부로 메타 러너 학습, 후반부로 평가
split = len(y_valid) // 2  # 검증셋을 절반으로 분할

# 3모델 예측값을 메타 러너 입력으로 구성 (열 방향 결합)
meta_X = np.___(  # Q11-1: 열 방향으로 배열 결합 함수
    [pred_xgb, pred_lgb, pred_cat]
)

# Ridge 메타 러너 학습 (검증셋 전반부로 학습)
meta_model = ___(alpha=1.0, ___=___)  # Q11-2,3,4: Ridge 클래스, positive 파라미터명, True
meta_model.___(meta_X[:split], y_valid[:split])  # Q11-5: 메타 러너 학습

In [ ]:
# 검증셋 후반부로 스태킹 성능 평가
pred_stack = meta_model.___(meta_X[split:])  # Q11-6: 메타 러너 예측
mae_stack  = mean_absolute_error(y_valid[split:], pred_stack)  # 스태킹 MAE 계산
# 방향(부호) 정확도: 실제와 예측의 부호가 같은 비율
acc_stack  = (np.___(y_valid[split:]) == np.___(pred_stack)).mean()  # Q11-7: 부호 함수

print(f'스태킹 MAE:         {mae_stack:.6f} ({mae_stack*100:.4f}%)')
print(f'스태킹 방향 정확도: {acc_stack*100:.1f}%')
print(f'\n메타 러너 가중치:')
for name, coef in zip(['XGB', 'LGB', 'CAT'], meta_model.coef_):
    print(f'  {name}: {coef:.4f}')

In [ ]:
# XGBoost 피처 중요도 시각화
importance = pd.Series(
    model_xgb.feature_importances_,  # XGBoost 내장 피처 중요도
    index=FEATURE_COLS  # 피처 이름을 인덱스로 설정
).sort_values(ascending=False)  # 중요도 내림차순 정렬

importance.plot(kind='bar', figsize=(12, 4), title='XGBoost Feature Importance')
plt.tight_layout()  # 레이아웃 자동 조정
plt.show()
print(importance)  # 중요도 수치 출력

## Q12. 방향성 정확도 평가 (빈칸 2개)
- 수익률의 부호(+/-)가 맞으면 방향 예측 성공
- 부호 함수: `np.sign()`


In [ ]:
# 방향성 예측 성능 테스트
# 검증 데이터의 실제 방향(부호) 추출
actual_dir = np.___(y_valid)  # Q12-1: 부호 함수
# 앙상블 예측의 방향(부호) 추출
pred_dir   = np.___(pred_ens)  # Q12-2: 부호 함수
# 방향이 일치하는 비율 = 정확도
accuracy   = (actual_dir == pred_dir).mean()
print(f'앙상블 방향 정확도: {accuracy*100:.1f}%')

**단순 XGBoost 모델 예측보다 XGBoost + LightGBM + CatBoost 앙상블 + 섹터 ETF 피처 추가 + Optuna 튜닝으로 방향성 예측 개선**

### 향후 30일 재귀 예측

매 스텝마다 예측된 수익률로 다음날 가격을 계산하고,
기술적 지표(MA괴리율·RSI·거래량비율)는 재계산합니다.
외부 변수(VIX·금리·섹터 ETF)는 최근 1년 평균으로 고정합니다.

## Q13. 재귀 예측 설정 (빈칸 3개)
- 예측 기간 = ?일
- 외부 변수 고정 기간 = 최근 ?일 (약 1년)


In [ ]:
PREDICT_DAYS = ___  # Q13-1: 향후 예측 기간 (영업일)

# 외부 변수 고정값 (최근 1년 평균으로 미래 외부변수 대체)
vix_mean       = df['vix'].iloc[-___:].mean()  # Q13-2: 최근 1년 VIX 평균
tnx_mean       = df['tnx'].iloc[-___:].mean()  # Q13-3: 최근 1년 금리 평균
yield_mean     = df['yield_spread'].iloc[-252:].mean()  # 최근 1년 금리차 평균
dxy_mean       = df['dxy_return'].iloc[-252:].mean()  # 최근 1년 달러 수익률 평균
vol_ratio_mean = df['sp500_vol_ratio'].iloc[-252:].mean()  # 최근 1년 거래량 비율 평균

In [ ]:
# 섹터 ETF 수익률 평균값 (최근 1년)
sector_means = {
    'xlf_return':   df['xlf_return'].iloc[-252:].mean(),  # 금융 섹터 수익률 평균
    'xle_return':   df['xle_return'].iloc[-252:].mean(),  # 에너지 섹터 수익률 평균
    'xlk_return':   df['xlk_return'].iloc[-252:].mean(),  # 기술 섹터 수익률 평균
    'xlv_return':   df['xlv_return'].iloc[-252:].mean(),  # 헬스케어 섹터 수익률 평균
    'xlk_sp500_rs': df['xlk_sp500_rs'].iloc[-252:].mean(),  # 기술주 상대강도 평균
}

# 예측 날짜 사전 생성 (영업일 기준, 계절성 피처 계산용)
future_dates = pd.bdate_range(
    start=df.index[-1] + pd.Timedelta(days=1),  # 마지막 데이터 다음 날부터
    periods=PREDICT_DAYS  # 예측 기간만큼
)

print(f'고정 외부변수 | VIX: {vix_mean:.2f}, TNX: {tnx_mean:.2f}, 금리차: {yield_mean:.2f}')
print(f'예측 기간: {future_dates[0].date()} ~ {future_dates[-1].date()}')

## Q14-15. 재귀 예측 루프 (빈칸 11개)
- 20일 수익률: `pct_change(?)`
- MA 괴리율: 현재가 / MA20
- 라그 인덱스: `iloc[-?]`
- MA200 시장 레짐 판별
- 스케일러 변환: `scaler.transform(feat)`
- 다음 가격 = 현재가 × (1 + 예측 수익률)
- **힌트**: 20, `ma20`, 2, 5, 200, `transform`, `predict`, `pred_ret`, `next_price`

In [ ]:
# 최근 가격 시계열 (기술 지표 재계산용 버퍼)
price_buf = list(df['sp500_close'].values)  # 기존 가격 데이터를 리스트로 복사
last_price = price_buf[-1]  # 마지막(현재) 가격

predicted_prices = [last_price]  # 예측 가격 저장 리스트 (현재가부터 시작)

for step in range(PREDICT_DAYS):
    prices    = pd.Series(price_buf)  # 가격 버퍼를 시리즈로 변환
    returns_s = prices.pct_change(1)  # 일별 수익률 계산

    # 기술적 지표 계산
    ret20     = prices.pct_change(___).iloc[-1]  # Q14-1: 20일 누적 수익률
    ma20      = prices.rolling(___).mean().iloc[-1]  # Q14-2: 20일 이동평균
    ma20_disp = prices.iloc[-1] / ___ if ___ > 0 else 1.0  # Q14-3,4: MA20 괴리율 (현재가/MA20)

    # 라그 피처 (가격 버퍼에서 직접 계산)
    ret_lag2 = returns_s.iloc[-___] if len(returns_s) >= 3 else 0.0  # Q15-1: 2일 전 수익률 인덱스
    ret_lag5 = returns_s.iloc[-___] if len(returns_s) >= 6 else 0.0  # Q15-2: 5일 전 수익률 인덱스

    # 변동성 피처 (롤링 표준편차)
    vol_20d_val = returns_s.iloc[-20:].std()  # 최근 20일 수익률 변동성
    vol_60d_val = returns_s.iloc[-60:].std()  # 최근 60일 수익률 변동성

    # 계절성 피처 (예측 날짜 기준)
    cur_date = future_dates[step]  # 현재 예측 날짜
    dow_val  = cur_date.dayofweek  # 요일 (0=월~4=금)
    mon_val  = cur_date.month  # 월 (1~12)

    # 시장 레짐 판별 (MA200 기준)
    ma200    = prices.rolling(___).mean().iloc[-1]  # Q15-3: 200일 이동평균
    bull_val = 1 if prices.iloc[-1] > ma200 else 0  # 현재가 > MA200 이면 상승장

    # 피처 벡터 구성 (FEATURE_COLS 순서와 일치: 21개)
    feat = np.array([[
        ret20, ma20_disp, vol_ratio_mean,  # 수익률, MA괴리율, 거래량비율
        vix_mean, tnx_mean, yield_mean, dxy_mean,  # 외부변수 (고정값)
        sector_means['xlf_return'], sector_means['xle_return'],  # 섹터 수익률 (고정값)
        sector_means['xlk_return'], sector_means['xlv_return'],  # 섹터 수익률 (고정값)
        sector_means['xlk_sp500_rs'],  # 기술주 상대강도 (고정값)
        vix_mean, vix_mean,  # vix_lag2, vix_lag5 (고정값)
        ret_lag2, ret_lag5,  # 라그 피처 (재계산)
        vol_20d_val, vol_60d_val,  # 변동성 피처 (재계산)
        dow_val, mon_val, bull_val,  # 계절성 + 레짐 (재계산)
    ]])
    feat_s = scaler.___(feat)  # Q15-4: 학습 스케일러로 정규화

    # 3모델 예측 → 메타 러너 앙상블
    p_xgb = float(model_xgb.___(feat_s)[0])  # Q15-5: XGBoost 수익률 예측
    p_lgb = float(model_lgb.predict(feat_s)[0])  # LightGBM 수익률 예측
    p_cat = float(model_cat.predict(feat_s)[0])  # CatBoost 수익률 예측
    # 메타 러너가 3모델 예측을 최적 결합하여 최종 수익률 산출
    pred_ret = float(meta_model.predict([[p_xgb, p_lgb, p_cat]])[0])

    # 다음날 가격 = 현재가 × (1 + 예측 수익률)
    next_price = price_buf[-1] * (1 + ___)  # Q15-6: 예측 수익률 변수명
    predicted_prices.append(next_price)  # 예측 가격 저장
    price_buf.append(___)  # Q15-7: 버퍼에 추가할 가격 변수

print(f'현재가: {last_price:.2f}')
print(f'30일 후 예측가: {predicted_prices[-1]:.2f}')
print(f'예상 수익률: {(predicted_prices[-1]/last_price - 1)*100:.2f}%')

### 예측 결과 시각화

In [ ]:
# 최근 60일 실제 + 30일 예측 시각화
recent_prices = df['sp500_close'].iloc[-60:]  # 최근 60일 종가
recent_dates  = df.index[-60:]  # 최근 60일 날짜

# 예측 날짜 생성 (영업일 기준)
last_date = df.index[-1]  # 마지막 실제 데이터 날짜
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=PREDICT_DAYS)

fig, ax = plt.subplots(figsize=(12, 5))  # 그래프 크기 설정
ax.plot(recent_dates, recent_prices, color='#1C1C1E', linewidth=1.5, label='실제')  # 실제 가격
ax.plot(future_dates, predicted_prices[1:], color='#007AFF', linewidth=1.5,
        linestyle='--', label='앙상블 예측')  # 예측 가격 (점선)
ax.axvline(last_date, color='#8E8E93', linewidth=0.8, linestyle=':')  # 현재 시점 표시 (세로선)
ax.set_title(f'S&P 500 향후 {PREDICT_DAYS}일 예측 (앙상블)', fontsize=13)  # 그래프 제목
ax.set_ylabel('지수')  # Y축 레이블
ax.legend()  # 범례 표시
ax.grid(alpha=0.2)  # 격자선 (투명도 0.2)
plt.tight_layout()  # 레이아웃 자동 조정
plt.show()

---
## 정답표

| # | 빈칸 | 정답 | 설명 |
|---|------|------|------|
| Q1-1 | `from xgboost import ___` | `XGBRegressor` | XGBoost 회귀 모델 클래스 |
| Q1-2 | `from lightgbm import ___` | `LGBMRegressor` | LightGBM 회귀 모델 클래스 |
| Q1-3 | `from catboost import ___` | `CatBoostRegressor` | CatBoost 회귀 모델 클래스 |
| Q1-4 | `from sklearn.preprocessing import ___` | `StandardScaler` | 표준 정규화 스케일러 |
| Q1-5 | `from sklearn.metrics import ___` | `mean_absolute_error` | 평균 절대 오차 함수 |
| Q1-6 | `from sklearn.linear_model import ___` | `Ridge` | 릿지 회귀 (메타 러너) |
| Q2-1 | `pd.to_datetime(..., unit='___')` | `s` | 유닉스 타임스탬프(초) 단위 |
| Q3-1 | `pct_change(___)` | `5` | 5일 누적 수익률 |
| Q3-2 | `pct_change(___)` | `20` | 20일 누적 수익률 |
| Q3-3 | `pct_change(___)` | `1` | 1일 수익률 |
| Q3-4 | `pct_change(___)` (XLK) | `5` | XLK 5일 수익률 |
| Q3-5 | `pct_change(___)` (S&P500) | `5` | S&P500 5일 수익률 |
| Q3-6 | `shift(___)` | `-1` | 다음날 수익률을 현재로 당김 |
| Q4-1 | `shift(___)` (vix_lag2) | `2` | VIX 2일 전 값 |
| Q4-2 | `shift(___)` (vix_lag5) | `5` | VIX 5일 전 값 |
| Q4-3 | `shift(___)` (ret_lag2) | `2` | 수익률 2일 전 값 |
| Q4-4 | `shift(___)` (ret_lag5) | `5` | 수익률 5일 전 값 |
| Q4-5 | `rolling(___)` | `20` | 20일 롤링 윈도우 |
| Q4-6 | `.___()` | `std` | 표준편차 |
| Q4-7 | `rolling(___)` | `60` | 60일 롤링 윈도우 |
| Q4-8 | `.___()` | `std` | 표준편차 |
| Q4-9 | `rolling(___)` | `200` | 200일 이동평균 |
| Q4-10 | `.___()` | `mean` | 이동평균 |
| Q5-1 | `.___()` (평균) | `mean` | 타깃 평균 계산 |
| Q5-2 | `.___()` (표준편차) | `std` | 타깃 표준편차 계산 |
| Q5-3 | `___*ret_std` (하한) | `3` | 3σ 클리핑 |
| Q5-4 | `___*ret_std` (상한) | `3` | 3σ 클리핑 |
| Q6-1 | `VALID_DAYS = ___` | `252` | 약 1년 영업일 |
| Q6-2 | `___()` (스케일러) | `StandardScaler` | 표준 정규화 |
| Q6-3 | `scaler.___()` | `fit_transform` | 학습 + 변환 |
| Q6-4 | `scaler.___()` | `transform` | 변환만 |
| Q6-5 | `decay = ___` | `0.9995` | 지수 감쇠 계수 |
| Q7-1 | `m.___(...)` | `fit` | 모델 학습 (XGB) |
| Q7-2 | `___(y_valid, ...)` | `mean_absolute_error` | MAE 함수 (XGB) |
| Q7-3 | `m.___(X_valid_s)` | `predict` | 모델 예측 (XGB) |
| Q7-4 | `m.___(...)` | `fit` | 모델 학습 (LGB) |
| Q7-5 | `___(y_valid, m.___(X_valid_s))` | `mean_absolute_error`, `predict` | MAE 계산 (LGB) |
| Q7-6 | `m.___(...)` | `fit` | 모델 학습 (CAT) |
| Q7-7 | `___(y_valid, m.___(X_valid_s))` | `mean_absolute_error`, `predict` | MAE 계산 (CAT) |
| Q8-1 | `direction='___'` | `minimize` | MAE 최소화 (XGB) |
| Q8-2 | `direction='___'` | `minimize` | MAE 최소화 (LGB) |
| Q8-3 | `direction='___'` | `minimize` | MAE 최소화 (CAT) |
| Q9-1 | `___(**params)` | `XGBRegressor` | XGBoost 모델 생성 |
| Q9-2 | `model_xgb.___(...)` | `fit` | XGBoost 학습 |
| Q9-3 | `___(**params)` | `LGBMRegressor` | LightGBM 모델 생성 |
| Q9-4 | `model_lgb.___(...)` | `fit` | LightGBM 학습 |
| Q9-5 | `___(**params)` | `CatBoostRegressor` | CatBoost 모델 생성 |
| Q9-6 | `model_cat.___(...)` | `fit` | CatBoost 학습 |
| Q10-1 | `model_xgb.___(...)` | `predict` | XGBoost 예측 |
| Q10-2 | `model_lgb.___(...)` | `predict` | LightGBM 예측 |
| Q10-3 | `model_cat.___(...)` | `predict` | CatBoost 예측 |
| Q10-4 | `___(y_valid, pred_xgb)` | `mean_absolute_error` | XGBoost MAE |
| Q10-5 | `___(y_valid, pred_lgb)` | `mean_absolute_error` | LightGBM MAE |
| Q10-6 | `___(y_valid, pred_cat)` | `mean_absolute_error` | CatBoost MAE |
| Q10-7 | `___ / mae_xgb` | `1` | MAE 역수 (분자=1) |
| Q10-8 | `___ / mae_lgb` | `1` | MAE 역수 (분자=1) |
| Q10-9 | `___ / mae_cat` | `1` | MAE 역수 (분자=1) |
| Q11-1 | `np.___([...])` | `column_stack` | 열 방향 결합 |
| Q11-2 | `___()` (메타 러너) | `Ridge` | 릿지 회귀 |
| Q11-3 | `___=___` (파라미터명) | `positive` | 음수 가중치 방지 파라미터 |
| Q11-4 | `positive=___` | `True` | 음수 가중치 방지 활성화 |
| Q11-5 | `meta_model.___(...)` | `fit` | 메타 러너 학습 |
| Q11-6 | `meta_model.___(...)` | `predict` | 메타 러너 예측 |
| Q11-7 | `np.___(...)` | `sign` | 부호 함수 |
| Q12-1 | `np.___(y_valid)` | `sign` | 부호 함수 |
| Q12-2 | `np.___(pred_ens)` | `sign` | 부호 함수 |
| Q13-1 | `PREDICT_DAYS = ___` | `30` | 30일 예측 |
| Q13-2 | `iloc[-___:]` | `252` | 최근 1년 (VIX) |
| Q13-3 | `iloc[-___:]` | `252` | 최근 1년 (금리) |
| Q14-1 | `pct_change(___)` | `20` | 20일 수익률 |
| Q14-2 | `rolling(___)` | `20` | 20일 이동평균 |
| Q14-3 | `/ ___` | `ma20` | MA20 변수 |
| Q14-4 | `if ___ > 0` | `ma20` | MA20 > 0 체크 |
| Q15-1 | `iloc[-___]` | `2` | 2일 전 인덱스 |
| Q15-2 | `iloc[-___]` | `5` | 5일 전 인덱스 |
| Q15-3 | `rolling(___)` | `200` | 200일 이동평균 |
| Q15-4 | `scaler.___(feat)` | `transform` | 정규화 변환 |
| Q15-5 | `model_xgb.___(...)` | `predict` | XGBoost 예측 |
| Q15-6 | `(1 + ___)` | `pred_ret` | 예측 수익률 |
| Q15-7 | `price_buf.append(___)` | `next_price` | 예측 가격 버퍼 추가 |

**총 빈칸: 80개**